In [1]:
# %matplotlib inline
%config InlineBackend.figure_format = "retina"
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
})

import seaborn as sns
sns.set_theme(context="talk", style="whitegrid", 
              palette="colorblind", color_codes=True, 
              rc={"figure.figsize": [12, 8]})

import yfinance as yf
import numpy as np
import pandas as pd
import quantstats as qs

import requests
import csv

from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

import statsmodels.api as sm

from pypfopt import black_litterman
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier

In [2]:
# Carteira teorica do Idiv

url = "https://raw.githubusercontent.com/BDonadelli/Finance-playground/main/data/Cart_Idiv.csv"

response = requests.get(url)
response.encoding = 'utf-8'  # ou 'latin-1' se necessário

# Divide o conteúdo em linhas
linhas = response.text.splitlines()

# Ignora as duas primeiras e duas últimas linhas
linhas_filtradas = linhas[2:-2]

# Extrai a primeira coluna de cada linha restante
ASSETS = []
for linha in linhas_filtradas:
    # O separador é ";", pega o primeiro campo
    campos = linha.split(';')
    if campos:  # garante que a linha não está vazia
        ASSETS.append((campos[0],campos[4]))

ASSETS = [(ticker, float(str(value).replace(',', '.'))) for ticker, value in ASSETS]

try:
    pesos_indice = [value for ticker, value in ASSETS]
    tickers = [ticker+'.SA' for ticker, value in ASSETS]
    tickers.sort()
except :    
    tickers = [ticker+'.SA' for ticker in ASSETS]

tickers.append('AMBP3.SA')
tickers.sort()

print(tickers)

['ABCB4.SA', 'AGRO3.SA', 'ALOS3.SA', 'AMBP3.SA', 'BBAS3.SA', 'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BRAP4.SA', 'BRBI11.SA', 'BRSR6.SA', 'CMIG4.SA', 'CMIN3.SA', 'CPFE3.SA', 'CSMG3.SA', 'CURY3.SA', 'CXSE3.SA', 'DIRR3.SA', 'EGIE3.SA', 'EVEN3.SA', 'EZTC3.SA', 'FESA4.SA', 'FLRY3.SA', 'GRND3.SA', 'ISAE4.SA', 'ITSA4.SA', 'ITUB3.SA', 'ITUB4.SA', 'JHSF3.SA', 'KEPL3.SA', 'KLBN11.SA', 'LAVV3.SA', 'LEVE3.SA', 'LOGG3.SA', 'MBRF3.SA', 'ODPV3.SA', 'PETR3.SA', 'PETR4.SA', 'PGMN3.SA', 'POMO4.SA', 'RANI3.SA', 'RECV3.SA', 'SAPR11.SA', 'SLCE3.SA', 'SYNE3.SA', 'TAEE11.SA', 'TGMA3.SA', 'TIMS3.SA', 'UNIP6.SA', 'VALE3.SA', 'VBBR3.SA', 'VLID3.SA', 'VULC3.SA']


In [3]:
ASSETS = [
    "ABEV3.SA",
    "VLID3.SA",
    "OFSA3.SA",
    "TECN3.SA",
    "SOND5.SA",
    "MDNE3.SA",
    "CURY3.SA",
    "RANI3.SA",
    "JHSF3.SA",
    "LPSB3.SA",
    "MULT3.SA",
    "ITSA4.SA",
    "RECV3.SA",
    "CSUD3.SA",
    "LUXM4.SA",
    "FIQE3.SA",
    "BLAU3.SA",
    "SHUL4.SA",
    "EALT4.SA",
    "RSUL4.SA",
    "POMO4.SA",
    "PETR4.SA",
    "SBSP3.SA",
    "CSMG3.SA",
    "WIZC3.SA",
    "MILS3.SA",
    "CAMB3.SA",
    "VULC3.SA",
    "GRND3.SA"
]

#### Parâmetros

In [4]:
rf = 0.14               # taxa livre de risco
n_days=252              # dias no ano do calendario financeiro, assumindo dados diários pegos no Yahoo Finance
#n_monte_carlo = 10**6 # quantidade de carteiras na simulação

# Definição do período e download dos dados
data_inicio = '2018-01-01'
data_fim = '2026-04-30'

#### preços de fechamento

Baixa dados e limpa a base

In [5]:
prices = yf.download(tickers, start=data_inicio, end=data_fim , auto_adjust=True)['Close']
prices.columns = [col.replace('.SA', '') for col in prices.columns]

benchm = yf.download('^BVSP', start=data_inicio, end=data_fim , auto_adjust=True)['Close']

[*********************100%***********************]  53 of 53 completed
[*********************100%***********************]  1 of 1 completed


In [6]:
# Empresas com mais de 'limiar' dados faltantes
limiar = 10
missing = prices.isna().sum()
empresas_missing = missing[missing > limiar].index.tolist()

print(missing[missing > limiar].sort_values(ascending=False))

tickers = [item for item in ASSETS if item not in empresas_missing]



BRBI11    872
RECV3     824
CXSE3     821
CMIN3     774
CURY3     674
LAVV3     663
PGMN3     663
AMBP3     625
ALOS3     394
LOGG3     242
VBBR3      49
dtype: int64


In [7]:
import plotly.express as px

if empresas_missing:
    fig = px.line(
        prices[empresas_missing].reset_index(),
        x='Date',
        y=empresas_missing,
        title='Empresas com mais de 10 dados faltantes',
        labels={'value': 'Preço', 'Date': 'Data', 'variable': 'Empresa'}
    )

    fig.show()

In [8]:
# mantem colunas (axis=1) ue possuem no mínimo len(prices) - 20 valores não-nulos.
prices = prices.dropna(axis=1, thresh=len(prices) - 10)
# preenche dados faltantes repetindo ultimo valor
prices = prices.ffill()

prices

,ABCB4,AGRO3,BBAS3,BBDC3,BBDC4,BBSE3,BRAP4,BRSR6,CMIG4,CPFE3,...,SAPR11,SLCE3,SYNE3,TAEE11,TGMA3,TIMS3,UNIP6,VALE3,VLID3,VULC3
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,9.208961,6.842369,9.456818,11.138395,12.179823,13.367785,4.202160,7.811342,1.498885,12.097425,...,12.398586,3.451246,1.566560,9.288570,11.868076,7.978257,5.058344,21.669313,11.916983,5.375476
2018-01-03,9.247842,6.836967,9.577435,11.185783,12.235812,13.377099,4.241884,7.848086,1.485963,11.947997,...,12.291394,3.481202,1.650874,9.301553,11.868076,7.984297,5.428319,21.539465,12.096980,5.641647
2018-01-04,9.220069,7.058386,9.669329,11.389478,12.436576,13.405046,4.362473,7.895333,1.468734,11.860832,...,12.192608,3.467472,1.720012,9.154392,12.324993,7.948032,5.370699,21.627758,12.096980,5.786303
2018-01-05,9.353372,7.204199,9.669329,11.392924,12.507018,13.493545,4.467455,7.998995,1.470888,11.667821,...,12.285088,3.532378,1.720012,9.197677,12.461477,8.014518,5.455612,21.965368,12.072154,5.786303
2018-01-08,9.442240,7.236600,9.692304,11.392924,12.503497,13.572726,4.545483,8.132941,1.475195,11.854605,...,12.211523,3.513655,1.736874,9.154392,12.461477,7.905725,5.701250,22.453606,12.041121,5.815236
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,25.160000,19.680000,23.000000,17.191864,19.949961,34.270000,24.059999,16.155130,13.088566,50.148193,...,42.849998,17.530001,4.060000,43.116379,32.939999,26.049999,60.869999,85.970001,19.564034,16.120001
2026-04-24,25.280001,19.850000,22.700001,17.091970,19.900011,34.290001,23.950001,15.876423,12.804031,49.388653,...,43.610001,17.370001,4.000000,42.585388,32.230000,26.100000,60.540001,85.870003,19.653141,15.940000
2026-04-27,24.930000,19.139999,22.510000,16.952118,19.710201,33.950001,23.850000,15.687300,12.558743,48.906994,...,41.549999,17.240000,3.940000,42.102669,32.250000,25.770000,59.880001,85.500000,19.247208,15.780000


In [9]:
returns = prices.pct_change().dropna()

#log retorno para matriz de covriancia ledoit wolf, tomar cuidado com parametros das bibliotecas

print(len(prices),len(returns),len(tickers),len(ASSETS),len(prices.columns))

2068 2067 29 29 42


Matriz covariância por Ledoit Wolf

In [18]:
from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

"""
cov_matrix é uma NxN matriz de covariância
tickers é um dicionário com os ativos
"""

pesos = [1 / len(ASSETS)] * len(ASSETS)
# dicionario_pesos = dict(zip(tickers, pesos))

#cov = np.cov(market_prices)

model = LedoitWolf()
cov_matrix = model.fit(returns).covariance_
df_cov_matrix = pd.DataFrame(cov_matrix, index=returns.columns, columns=returns.columns)

print(df_cov_matrix.shape)
#print(dicionario_pesos)
print(tickers)

tickers_validos = list(df_cov_matrix.columns)

pesos = np.repeat(1/len(tickers_validos), len(tickers_validos))

dicionario_pesos = dict(zip(tickers_validos, pesos))



(42, 42)
['ABEV3.SA', 'VLID3.SA', 'OFSA3.SA', 'TECN3.SA', 'SOND5.SA', 'MDNE3.SA', 'CURY3.SA', 'RANI3.SA', 'JHSF3.SA', 'LPSB3.SA', 'MULT3.SA', 'ITSA4.SA', 'RECV3.SA', 'CSUD3.SA', 'LUXM4.SA', 'FIQE3.SA', 'BLAU3.SA', 'SHUL4.SA', 'EALT4.SA', 'RSUL4.SA', 'POMO4.SA', 'PETR4.SA', 'SBSP3.SA', 'CSMG3.SA', 'WIZC3.SA', 'MILS3.SA', 'CAMB3.SA', 'VULC3.SA', 'GRND3.SA']


In [19]:
delta = black_litterman.market_implied_risk_aversion(prices)
prior = black_litterman.market_implied_prior_returns(
    dicionario_pesos,
    delta,
    cov_matrix
)
print(delta)

ABCB4     1.777735
AGRO3     2.011665
BBAS3     1.300449
BBDC3     0.932729
BBDC4     0.966698
BBSE3     2.321506
BRAP4     2.204131
BRSR6     1.226214
CMIG4     2.666634
CPFE3     2.861778
CSMG3     2.605378
DIRR3     2.237331
EGIE3     2.913167
EVEN3     0.752133
EZTC3     0.630332
FESA4     1.167987
FLRY3     0.321384
GRND3     0.549470
ISAE4     3.358210
ITSA4     2.427836
ITUB3     2.534813
ITUB4     1.915719
JHSF3     1.851543
KEPL3     1.938176
KLBN11    1.258539
LEVE3     1.699175
MBRF3     1.171516
ODPV3     0.967243
PETR3     2.150487
PETR4     2.226397
POMO4     1.389336
RANI3     1.247864
SAPR11    2.085161
SLCE3     2.054291
SYNE3     0.870921
TAEE11    5.249978
TGMA3     1.234115
TIMS3     2.061352
UNIP6     2.270381
VALE3     1.759531
VLID3     0.749243
VULC3     1.335835
dtype: float64


/home/caio/Documentos/ic_26/.venv/lib/python3.12/site-packages/pypfopt/black_litterman.py:45: RuntimeWarning: If cov_matrix is not a dataframe, market cap index must be aligned to cov_matrix
  warnings.warn(


In [20]:
print(prior)

ABCB4     0.000307
AGRO3     0.000203
BBAS3     0.000289
BBDC3     0.000194
BBDC4     0.000203
BBSE3     0.000270
BRAP4     0.000312
BRSR6     0.000235
CMIG4     0.000534
CPFE3     0.000383
CSMG3     0.000447
DIRR3     0.000516
EGIE3     0.000338
EVEN3     0.000215
EZTC3     0.000176
FESA4     0.000180
FLRY3     0.000053
GRND3     0.000088
ISAE4     0.000349
ITSA4     0.000429
ITUB3     0.000415
ITUB4     0.000345
JHSF3     0.000476
KEPL3     0.000247
KLBN11    0.000106
LEVE3     0.000267
MBRF3     0.000205
ODPV3     0.000109
PETR3     0.000466
PETR4     0.000479
POMO4     0.000286
RANI3     0.000195
SAPR11    0.000314
SLCE3     0.000213
SYNE3     0.000163
TAEE11    0.000493
TGMA3     0.000279
TIMS3     0.000284
UNIP6     0.000419
VALE3     0.000247
VLID3     0.000161
VULC3     0.000277
dtype: float64


In [21]:
fatores_nefin = pd.read_csv('nefin_factors.csv')

# A coluna Date já existe, só converter e setar como índice
fatores_nefin['Date'] = pd.to_datetime(fatores_nefin['Date'])
fatores_nefin.set_index('Date', inplace=True)

# Reamostrar os fatores diários para mensais
#fatores_mensais = fatores_nefin.resample('ME').sum()
#print(fatores_mensais)

fatores_mensais = (1 + fatores_nefin).resample('ME').prod() - 1
print(fatores_mensais)
#verificar se a taxa e fatores estao sendo colocadas certas mensalmente

# Alinhar com os retornos
datas_comuns = returns.index.intersection(fatores_mensais.index)
retornos_alinhados = returns.loc[datas_comuns]
fatores_alinhados = fatores_mensais.loc[datas_comuns]

                     Unnamed: 0  Rm_minus_Rf       SMB       HML       WML  \
Date                                                                         
2001-01-31 -1250660718674968577     0.139540  0.163653  0.147510 -0.012725   
2001-02-28   231432332370247679    -0.085317  0.052359  0.022571  0.071785   
2001-03-31 -2890597822505680897    -0.077331 -0.014624  0.060201  0.076311   
2001-04-30  7254507651604676607     0.026857 -0.120024 -0.155679 -0.049503   
2001-05-31  8983196893043490815    -0.003461 -0.094637 -0.154272 -0.017923   
...                         ...          ...       ...       ...       ...   
2025-12-31 -8644949316997480449     0.004598 -0.031421  0.009458 -0.028194   
2026-01-31  4353516797392060415     0.103267 -0.014013  0.043407  0.023017   
2026-02-28  3190475155598606335     0.026488 -0.044621 -0.033540  0.007398   
2026-03-31 -8315929580154126337    -0.021942 -0.039934  0.004906 -0.021615   
2026-04-30             39181339     0.001203  0.005243  0.009763

In [22]:
print(pesos)

[0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952]


In [24]:
lista_tickers = tickers_validos

# -------------------------------------------------------
# ETAPA 1: OLS — regressão Fama-French por ativo
col_rf       = 'Risk_Free'
fatores_cols = [c for c in fatores_alinhados.columns if c != col_rf]
X_ols         = sm.add_constant(fatores_alinhados[fatores_cols])
medias_fatores = fatores_alinhados[fatores_cols].mean()

betas_dict = {}
for ticker in lista_tickers:
    Y      = retornos_alinhados[ticker] - fatores_alinhados[col_rf]
    modelo = sm.OLS(Y, X_ols).fit()
    betas_dict[ticker] = modelo.params

betas_df = pd.DataFrame(betas_dict).T

# ETAPA 2: Retornos estimados r̂_i (Eq. 12)
retornos_estimados = (
    betas_df['const']
    + betas_df[fatores_cols].dot(medias_fatores)
)

# ETAPA 3: Grid 5x5
book_to_market = {}
for ticker in lista_tickers:
    info = yf.Ticker(ticker + '.SA').info
    pb   = info.get('priceToBook', None)
    book_to_market[ticker] = 1 / pb if pb else None
book_to_market = pd.Series(book_to_market).dropna()

# ✅ market_cap criado ANTES de filtrar
market_cap = pd.Series(
    {ticker: peso for ticker, peso in zip(lista_tickers, pesos)}
)

# ✅ filtrar AMBOS para tickers com BM disponível
tickers_bm     = [t for t in lista_tickers if t in book_to_market.index]
market_cap     = market_cap[tickers_bm]      # ← corrigido (era market_cap[tickers_bm] sem definir antes)
book_to_market = book_to_market[tickers_bm]

# ✅ size_quintil calculado sobre market_cap já filtrado
n            = 5
size_quintil = pd.qcut(market_cap.rank(method='first'), n, labels=False)
bm_labels    = pd.Series(index=tickers_bm, dtype=int)

for g in range(n):
    grupo_tickers = size_quintil[size_quintil == g].index  # ← agora alinhado com tickers_bm
    bm_labels[grupo_tickers] = pd.qcut(
        book_to_market[grupo_tickers].rank(method='first'),
        n, labels=False, duplicates='drop'
    )

grupo_1  = [t for t in tickers_bm if size_quintil[t] == 0 and bm_labels[t] == 4]
grupo_25 = [t for t in tickers_bm if size_quintil[t] == 4 and bm_labels[t] == 0]

if not grupo_1 or not grupo_25:
    raise ValueError("Grupo 1 ou 25 vazio — verifique os dados de size/BM")

# ETAPA 4: P_t (Eq. 11)
P_t           = pd.Series(0.0, index=tickers_bm)
P_t[grupo_1]  =  1 / len(grupo_1)
P_t[grupo_25] = -1 / len(grupo_25)

# ETAPA 5: q_t (Eq. 13-15)
q_t = retornos_estimados[grupo_1].mean() - retornos_estimados[grupo_25].mean()
print(f"View return q_t: {q_t:.4f}")
print(f"Grupo 1  (small/value): {grupo_1}")
print(f"Grupo 25 (big/growth):  {grupo_25}")

# ETAPA 6: Ω_t (Eq. 16)

# -------------------------------------------------------
# ETAPA 6: Incerteza da visão Ω_t (Eq. 16)  τ = 0.1
# -------------------------------------------------------
# tau     = 0.1
# Sigma   = df_cov_matrix.loc[tickers_bm, tickers_bm].values
# p1      = P_t.values.reshape(1, -1)
# omega_t = float(tau * (p1 @ Sigma @ p1.T))

# # -------------------------------------------------------
# # ETAPA 7: Retornos posteriores Black-Litterman (Eq. 7-8)
# # -------------------------------------------------------
# pi      = prior[tickers_bm].values.reshape(-1, 1)
# P       = p1
# q       = np.array([[q_t]])
# Omega   = np.array([[omega_t]])

# Sigma_BL = np.linalg.inv(
#     np.linalg.inv(tau * Sigma) + P.T @ np.linalg.inv(Omega) @ P
# )

# mu_BL = Sigma_BL @ (
#     np.linalg.inv(tau * Sigma) @ pi
#     + P.T @ np.linalg.inv(Omega) @ q
# )

# mu_BL_series = pd.Series(mu_BL.flatten(), index=tickers_bm)
# print(mu_BL_series.sort_values(ascending=False))



View return q_t: 0.0002
Grupo 1  (small/value): ['BBAS3', 'BRSR6']
Grupo 25 (big/growth):  ['TIMS3', 'UNIP6']


In [27]:
tau     = 0.1
Sigma   = df_cov_matrix.loc[tickers_bm, tickers_bm].values
p1      = P_t.values.reshape(1, -1)


# ✅ Correto — .item() extrai o escalar de um array (1,1)
omega_t = (tau * (p1 @ Sigma @ p1.T)).item()

# Verificação de sanidade
print(f"q_t    : {q_t:.6f}")
print(f"omega_t: {omega_t:.6f}")
print(f"P_t não-zero: {P_t[P_t != 0]}")

# Preparar para BlackLittermanModel
P_matrix     = p1                        # (1, N)
Q_vector     = np.array([q_t])           # (1,)
Omega_matrix = np.array([[omega_t]])     # (1, 1)
cov_matrix_bm = df_cov_matrix.loc[tickers_bm, tickers_bm]
prior_bm      = prior[tickers_bm]

# Black-Litterman
bl = BlackLittermanModel(
    cov_matrix_bm,
    pi=prior_bm,
    Q=Q_vector,
    P=P_matrix,
    omega=Omega_matrix,
    tau=tau
)

mu_BL = bl.bl_returns()
print("\nRetornos posteriores BL:")
print(mu_BL.sort_values(ascending=False))

ef = EfficientFrontier(mu_BL, cov_matrix_bm)
ef.max_sharpe()
weights = ef.clean_weights()
print("\nPesos ótimos (max Sharpe):")
print(pd.Series(weights).sort_values(ascending=False))

q_t    : 0.000223
omega_t: 0.000036
P_t não-zero: BBAS3    0.5
BRSR6    0.5
TIMS3   -0.5
UNIP6   -0.5
dtype: float64

Retornos posteriores BL:
CMIG4     0.000569
DIRR3     0.000544
PETR4     0.000520
PETR3     0.000504
TAEE11    0.000503
JHSF3     0.000500
ITSA4     0.000475
CSMG3     0.000466
ITUB3     0.000457
CPFE3     0.000399
ITUB4     0.000397
BBAS3     0.000383
ISAE4     0.000357
EGIE3     0.000351
ABCB4     0.000340
SAPR11    0.000330
BRSR6     0.000326
UNIP6     0.000325
BRAP4     0.000317
POMO4     0.000305
TGMA3     0.000304
VULC3     0.000295
BBSE3     0.000292
LEVE3     0.000273
BBDC4     0.000259
VALE3     0.000253
KEPL3     0.000252
TIMS3     0.000250
BBDC3     0.000250
EVEN3     0.000248
EZTC3     0.000207
SLCE3     0.000207
MBRF3     0.000204
AGRO3     0.000203
RANI3     0.000197
SYNE3     0.000184
FESA4     0.000181
VLID3     0.000176
ODPV3     0.000121
GRND3     0.000101
KLBN11    0.000094
FLRY3     0.000070
dtype: float64

Pesos ótimos (max Sharpe):
TAEE11    0.7055

In [26]:
tau     = 0.1
Sigma   = df_cov_matrix.loc[tickers_bm, tickers_bm].values
p1      = P_t.values.reshape(1, -1)
omega_t = float(tau * (p1 @ Sigma @ p1.T))


# P_t: matriz de visão (1 x N) — já calculado antes
P_matrix = P_t[tickers_bm].values.reshape(1, -1)  # shape (1, N)

# q_t: retorno esperado dessa visão relativa (escalar)
# q_t = média(r̂ grupo1) - média(r̂ grupo25)
Q_vector = np.array([q_t])                         # shape (1,)

# Ω_t: incerteza da visão (1x1) — já calculado antes
Omega_matrix = np.array([[omega_t]])               # shape (1, 1)

# Covariância e prior filtrados para tickers_bm
cov_matrix_bm = df_cov_matrix.loc[tickers_bm, tickers_bm]
prior_bm      = prior[tickers_bm]

viewdict = {}
for t in grupo_1:
    viewdict[t] =  retornos_estimados[t]   # visão positiva (small/value)
for t in grupo_25:
    viewdict[t] =  retornos_estimados[t]   # visão negativa (big/growth)

# Matriz de covariância filtrada para tickers_bm
cov_matrix_bm = df_cov_matrix.loc[tickers_bm, tickers_bm]

# BlackLittermanModel com Ω customizado (calculado via Eq. 16 do artigo)
bl = BlackLittermanModel(
    cov_matrix_bm,
    pi=prior[tickers_bm],          # prior já calculado
    absolute_views=viewdict,
    omega=np.array([[omega_t]]),   # Ω_t da Eq. 16
    tau=tau
)

# Retornos posteriores μ_BL
mu_BL = bl.bl_returns()
print("\nRetornos posteriores BL:")
print(mu_BL.sort_values(ascending=False))

# Pesos ótimos
ef = EfficientFrontier(mu_BL, cov_matrix_bm)
ef.max_sharpe()
weights = ef.clean_weights()
print("\nPesos ótimos:")
print(pd.Series(weights).sort_values(ascending=False))

# Alternativamente, pesos implícitos pelo delta
bl.bl_weights(delta)
weights_bl = bl.clean_weights()
print("\nPesos BL implícitos pelo delta:")
print(pd.Series(weights_bl).sort_values(ascending=False))

TypeError: only 0-dimensional arrays can be converted to Python scalars

In [ ]:

# col_rf = 'Risk_Free'

# retornos_esperados_ff = pd.Series(index=retornos_alinhados.columns, dtype=float)

# X = sm.add_constant(fatores_alinhados)  # Fatores + intercepto
# medias_fatores = fatores_alinhados.drop(columns=[col_rf]).mean()
# rf_media = fatores_alinhados[col_rf].mean()

# for ticker in retornos_alinhados.columns:
#     # Y = excesso de retorno do ativo (retorno - taxa livre de risco)
#     Y = retornos_alinhados[ticker] - fatores_alinhados[col_rf]

#     # Regressão OLS
#     modelo = sm.OLS(Y, X).fit()

#     # r_esperado = rf + alpha + sum(beta_fator * media_fator)
#     fatores_cols = [c for c in fatores_alinhados.columns if c != col_rf]
#     retorno_estimado = (
#         rf_media
#         + modelo.params['const']
#         + sum(modelo.params[f] * medias_fatores[f] for f in fatores_cols)
#     )

#     retornos_esperados_ff[ticker] = retorno_estimado

# print(retornos_esperados_ff.sort_values(ascending=False))

FLRY3     0.008108
SYNE3     0.007862
TAEE11    0.007492
CMIG4     0.007475
FESA4     0.007435
BBSE3     0.007323
JHSF3     0.007318
ODPV3     0.007283
GRND3     0.007262
BRSR6     0.007162
CSMG3     0.007096
LEVE3     0.007066
EGIE3     0.007035
MBRF3     0.006942
AGRO3     0.006915
PETR4     0.006891
ISAE4     0.006890
KEPL3     0.006838
TGMA3     0.006812
DIRR3     0.006754
PETR3     0.006671
BRAP4     0.006661
CPFE3     0.006652
VALE3     0.006644
KLBN11    0.006625
EZTC3     0.006571
UNIP6     0.006507
ABCB4     0.006489
ITSA4     0.006415
SAPR11    0.006404
SLCE3     0.006344
EVEN3     0.006317
BBDC3     0.006223
ITUB3     0.006218
VLID3     0.006213
VULC3     0.006201
POMO4     0.006191
TIMS3     0.006167
ITUB4     0.006075
BBDC4     0.006026
RANI3     0.005980
BBAS3     0.005959
dtype: float64
